# Root-finding methods (Python — only `numpy` and `math`)

This notebook translates several root-finding methods and uses only **numpy** and **math** for numeric work and printing. Methods included:

- Bisection
- False position (Regula Falsi)
- Illinois
- Pegasus
- Secant

Example: `f(x) = x**5 - 2`. For each method we print a short summary and the first 10 iterations of the history.

In [1]:
import math
import numpy as np

def _ensure_callable(f):
    if not callable(f):
        raise TypeError("f must be a callable function of one variable")


In [2]:
def bisection(f, a, b, tol=1e-12, maxiter=100):
    _ensure_callable(f)
    fa, fb = f(a), f(b)
    if fa * fb > 0:
        raise ValueError("f(a) and f(b) must have opposite signs to bracket a root")
    history = []
    c = a
    for k in range(1, maxiter+1):
        c = (a + b) / 2.0
        fc = f(c)
        history.append((k, float(a), float(b), float(c), float(fc), float(abs(b - a))))
        if abs(fc) == 0 or abs(b - a)/2 < tol:
            return {"root": float(c), "iterations": k, "converged": True, "history": history}
        if fa * fc < 0:
            b, fb = c, fc
        else:
            a, fa = c, fc
    return {"root": float(c), "iterations": maxiter, "converged": False, "history": history}

In [3]:
def false_position(f, a, b, tol=1e-12, maxiter=100):
    _ensure_callable(f)
    fa, fb = f(a), f(b)
    if fa * fb > 0:
        raise ValueError("f(a) and f(b) must have opposite signs to bracket a root")
    history = []
    c = a
    for k in range(1, maxiter+1):
        c_old = c
        c = b - fb * (b - a) / (fb - fa)
        fc = f(c)
        history.append((k, float(a), float(b), float(c), float(fc), float(abs(c - c_old))))
        if abs(fc) == 0 or abs(c - c_old) < tol:
            return {"root": float(c), "iterations": k, "converged": True, "history": history}
        if fb * fc < 0:
            a, fa = b, fb
        else:
            fa = fc
        b, fb = c, fc
    return {"root": float(c), "iterations": maxiter, "converged": False, "history": history}

In [4]:
def illinois(f, a, b, tol=1e-12, maxiter=100):
    _ensure_callable(f)
    fa, fb = f(a), f(b)
    if fa * fb > 0:
        raise ValueError("f(a) and f(b) must have opposite signs to bracket a root")
    history = []
    c = a
    for k in range(1, maxiter+1):
        c_old = c
        c = b - fb * (b - a) / (fb - fa)
        fc = f(c)
        history.append((k, float(a), float(b), float(c), float(fc), float(abs(c - c_old))))
        if abs(fc) == 0 or abs(c - c_old) < tol:
            return {"root": float(c), "iterations": k, "converged": True, "history": history}
        if fb * fc < 0:
            a, fa = b, fb
        else:
            fa = fa / 2.0
        b, fb = c, fc
    return {"root": float(c), "iterations": maxiter, "converged": False, "history": history}

In [5]:
def pegasus(f, a, b, tol=1e-12, maxiter=100):
    _ensure_callable(f)
    fa, fb = f(a), f(b)
    if fa * fb > 0:
        raise ValueError("f(a) and f(b) must have opposite signs to bracket a root")
    history = []
    c = b
    for k in range(1, maxiter+1):
        c_old = c
        c = b - fb * (b - a) / (fb - fa)
        fc = f(c)
        history.append((k, float(a), float(b), float(c), float(fc), float(abs(c - c_old))))
        if abs(fc) == 0 or abs(c - c_old) < tol:
            return {"root": float(c), "iterations": k, "converged": True, "history": history}
        if fb * fc < 0:
            a, fa = b, fb
        else:
            fa = fa * fb / (fb + fc)
        b, fb = c, fc
    return {"root": float(c), "iterations": maxiter, "converged": False, "history": history}

In [6]:
def secant(f, x0, x1, tol=1e-12, maxiter=100):
    _ensure_callable(f)
    history = []
    x_prev, x_curr = float(x0), float(x1)
    f_prev, f_curr = f(x_prev), f(x_curr)
    for k in range(1, maxiter+1):
        if f_curr == f_prev:
            return {"root": float(x_curr), "iterations": k-1, "converged": False, "history": history, "message": "Division by zero in secant denominator"}
        x_next = x_curr - f_curr * (x_curr - x_prev) / (f_curr - f_prev)
        f_next = f(x_next)
        history.append((k, float(x_prev), float(x_curr), float(x_next), float(f_next), float(abs(x_next - x_curr))))
        if abs(x_next - x_curr) < tol:
            return {"root": float(x_next), "iterations": k, "converged": True, "history": history}
        x_prev, f_prev = x_curr, f_curr
        x_curr, f_curr = x_next, f_next
    return {"root": float(x_curr), "iterations": maxiter, "converged": False, "history": history}

In [7]:
# Example: f(x) = x**5 - 2
def f(x):
    return x**5 - 2

a, b = 0.5, 1.5
methods = {
    'bisection': lambda: bisection(f, a, b, tol=1e-14, maxiter=100),
    'false_position': lambda: false_position(f, a, b, tol=1e-14, maxiter=100),
    'illinois': lambda: illinois(f, a, b, tol=1e-14, maxiter=100),
    'pegasus': lambda: pegasus(f, a, b, tol=1e-14, maxiter=100),
    'secant': lambda: secant(f, a, b, tol=1e-14, maxiter=100)  # endpoints as secant guesses
}

results = {}
for name, fn in methods.items():
    try:
        results[name] = fn()
    except Exception as e:
        results[name] = {"error": str(e)}

# Print a simple summary (no pandas)
print("Summary:\n")
for name, res in results.items():
    if 'error' in res:
        print(f"{name}: ERROR -> {res['error']}")
    else:
        print(f"{name}: root={res['root']:.15g}, iterations={res['iterations']}, converged={res['converged']}")

Summary:

bisection: root=1.14869835499703, iterations=47, converged=True
false_position: ERROR -> float division by zero
illinois: root=1.14869835499703, iterations=12, converged=True
pegasus: root=1.14869835499704, iterations=10, converged=True
secant: root=1.14869835499704, iterations=12, converged=True


In [8]:
# Print first 10 history entries for each method
for name, res in results.items():
    print('\n' + '='*30)
    print(name.upper())
    if 'error' in res:
        print('Error:', res['error'])
        continue
    hist = res['history']
    print('First up to 10 iterations:')
    for h in hist[:10]:
        # each history row is (k, a, b, c, f(c), metric)
        print('iter {:3d}: a={:.12g}, b={:.12g}, c={:.12g}, f(c)={:.3e}, metric={:.3e}'.format(h[0], h[1], h[2], h[3], h[4], h[5]))


BISECTION
First up to 10 iterations:
iter   1: a=0.5, b=1.5, c=1, f(c)=-1.000e+00, metric=1.000e+00
iter   2: a=1, b=1.5, c=1.25, f(c)=1.052e+00, metric=5.000e-01
iter   3: a=1, b=1.25, c=1.125, f(c)=-1.980e-01, metric=2.500e-01
iter   4: a=1.125, b=1.25, c=1.1875, f(c)=3.614e-01, metric=1.250e-01
iter   5: a=1.125, b=1.1875, c=1.15625, f(c)=6.661e-02, metric=6.250e-02
iter   6: a=1.125, b=1.15625, c=1.140625, f(c)=-6.930e-02, metric=3.125e-02
iter   7: a=1.140625, b=1.15625, c=1.1484375, f(c)=-2.270e-03, metric=1.562e-02
iter   8: a=1.1484375, b=1.15625, c=1.15234375, f(c)=3.194e-02, metric=7.812e-03
iter   9: a=1.1484375, b=1.15234375, c=1.150390625, f(c)=1.478e-02, metric=3.906e-03
iter  10: a=1.1484375, b=1.150390625, c=1.1494140625, f(c)=6.238e-03, metric=1.953e-03

FALSE_POSITION
Error: float division by zero

ILLINOIS
First up to 10 iterations:
iter   1: a=0.5, b=1.5, c=0.760330578512, f(c)=-1.746e+00, metric=2.603e-01
iter   2: a=1.5, b=0.760330578512, c=0.936277160385, f(c)=-